In [29]:
import os
import json
import asyncio

import nest_asyncio
nest_asyncio.apply()

import boto3
from botocore.config import Config

AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

KNOWLEDGE_BASE_ID = os.getenv("BEDROCK_KB_ID", "AUAYMEH9NR")
GENERATION_MODEL_ARN = os.getenv(
    "BEDROCK_GENERATION_MODEL_ARN",
    f"arn:aws:bedrock:{AWS_REGION}::foundation-model/amazon.nova-micro-v1:0",
)
JUDGE_MODEL_ID = os.getenv("BEDROCK_JUDGE_MODEL_ID", "us.meta.llama3-1-8b-instruct-v1:0")
EMBEDDING_MODEL_ID = os.getenv("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")

boto_config = Config(connect_timeout=120, read_timeout=120, retries={"max_attempts": 2})
agent_client = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION, config=boto_config)
runtime_client = boto3.client("bedrock-runtime", region_name=AWS_REGION, config=boto_config)


In [30]:
def retrieve_contexts(question: str, number_of_results: int = 4) -> list[str]:
    response = agent_client.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": question},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": number_of_results,
            }
        },
    )

    contexts = []
    for result in response.get("retrievalResults", []):
        text = result.get("content", {}).get("text", "").strip()
        if text:
            contexts.append(text)
    return contexts


def answer_with_knowledge_base(question: str) -> str:
    response = agent_client.retrieve_and_generate(
        input={"text": question},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": KNOWLEDGE_BASE_ID,
                "modelArn": GENERATION_MODEL_ARN,
            },
        },
    )
    return response["output"]["text"]


question = "what is attention is all you need?"
print(answer_with_knowledge_base(question))
print(f"Retrieved {len(retrieve_contexts(question))} context chunks")


"Attention Is All You Need" is a research paper published by Google Brain in 2017. The paper proposes a new simple network architecture called the Transformer, which is based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. The Transformer is a model architecture that relies entirely on an attention mechanism to draw global dependencies between input and output. The authors claim that their models outperform previous models in machine translation tasks while being more parallelizable and requiring significantly less time to train. The Transformer has become a popular architecture in natural language processing and other fields due to its success in various tasks.
Retrieved 4 context chunks


In [31]:
dataset = [
    {
        "question": "What is self attention?",
        "ground_truth": "Self-attention lets each token in a sequence attend to other tokens in the same sequence to build contextual representations.",
    },
    {
        "question": "Who wrote Attention Is All You Need?",
        "ground_truth": "Attention Is All You Need was written by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, and Illia Polosukhin.",
    },
]


In [32]:
results = []

for row in dataset:
    question = row["question"]
    answer = answer_with_knowledge_base(question)
    contexts = retrieve_contexts(question)

    results.append(
        {
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": row["ground_truth"],
        }
    )

    print("QUESTION:", question)
    print("ANSWER:", answer[:500], "...")
    print("CONTEXT CHUNKS:", len(contexts))
    print()


QUESTION: What is self attention?
ANSWER: Self-attention, also known as intra-attention, is an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence. It is used in various tasks such as reading comprehension, abstractive summarization, textual entailment, and learning task-independent sentence representations. 

In the context of the Transformer model, self-attention is a key component that allows the model to compute representations of its input and output without using se ...
CONTEXT CHUNKS: 4

QUESTION: Who wrote Attention Is All You Need?
ANSWER: The paper "Attention Is All You Need" was written by a team of researchers from Google Brain and Google Research. The authors are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin. The paper proposes a new simple network architecture called the Transformer, which is based solely on attention mechanisms,

In [33]:
from datasets import Dataset

eval_dataset = Dataset.from_list(results)
eval_dataset


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})

In [34]:
from langchain_core.embeddings import Embeddings


class BedrockTitanEmbeddings(Embeddings):
    def __init__(self, client, model_id: str = EMBEDDING_MODEL_ID, dimensions: int | None = None, normalize: bool = True):
        self.client = client
        self.model_id = model_id
        self.dimensions = dimensions
        self.normalize = normalize

    def _embed_one(self, text: str) -> list[float]:
        body = {"inputText": text}
        if self.dimensions is not None:
            body["dimensions"] = self.dimensions
        if self.normalize is not None:
            body["normalize"] = self.normalize

        response = self.client.invoke_model(
            modelId=self.model_id,
            body=json.dumps(body),
            accept="application/json",
            contentType="application/json",
        )
        payload = json.loads(response["body"].read())
        return payload["embedding"]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed_one(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed_one(text)

    async def aembed_documents(self, texts: list[str]) -> list[list[float]]:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self.embed_documents, texts)

    async def aembed_query(self, text: str) -> list[float]:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self.embed_query, text)


In [35]:
from langchain_aws import ChatBedrock

judge_llm = ChatBedrock(
    model_id=JUDGE_MODEL_ID,
    provider="meta",
    region_name=AWS_REGION,
    model_kwargs={"max_gen_len": 2048},
)

embeddings = BedrockTitanEmbeddings(runtime_client)


C:\Users\Administrator\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\json\decoder.py:353: RuntimeWarning: coroutine 'Executor.wrap_callable_with_index.<locals>.wrapped_callable_async' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-107' coro=<as_completed.<locals>.sema_coro() done, defined at c:\Users\Administrator\Downloads\demo\.venv\Lib\site-packages\ragas\executor.py:35> wait_for=<Future pending cb=[Task.task_wakeup()]> cb=[as_completed.<locals>._on_completion() at C:\Users\Administrator\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\asyncio\tasks.py:602]>
Task was destroyed but it is pending!
task: <Task pending name='Task-108' coro=<as_completed.<locals>.sema_coro() done, defined at c:\Users\Administrator\Downloads\demo\.venv\Lib\site-packages\ragas\executor.py:35> wait_for=<Future pending cb=[Task.task_wakeup()]> cb=[as_completed.<locals>._on_completion() at C:\Users\Admin

In [36]:
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_recall, faithfulness
from ragas.run_config import RunConfig
import ragas.executor as ragas_executor
import threading


def _jupyter_safe_runner_init(
    self,
    jobs,
    desc,
    keep_progress_bar=True,
    raise_exceptions=True,
    run_config=None,
):
    threading.Thread.__init__(self)
    self.jobs = jobs
    self.desc = desc
    self.keep_progress_bar = keep_progress_bar
    self.raise_exceptions = raise_exceptions
    self.run_config = run_config or RunConfig()
    self.loop = None
    self.futures = None
    self.results = None


def _jupyter_safe_runner_run(self):
    self.loop = asyncio.new_event_loop()
    asyncio.set_event_loop(self.loop)
    self.futures = ragas_executor.as_completed(
        loop=self.loop,
        coros=[coro for coro, _ in self.jobs],
        max_workers=self.run_config.max_workers,
    )
    results = []
    try:
        results = self.loop.run_until_complete(self._aresults())
    finally:
        self.results = results
        self.loop.close()


ragas_executor.Runner.__init__ = _jupyter_safe_runner_init
ragas_executor.Runner.run = _jupyter_safe_runner_run

run_config = RunConfig(timeout=180, max_retries=2, max_wait=30, thread_timeout=240)

result = evaluate(
    eval_dataset,
    metrics=[faithfulness, context_recall, answer_relevancy],
    llm=judge_llm,
    embeddings=embeddings,
    is_async=False,
    run_config=run_config,
)

result


Evaluating: 100%|██████████| 6/6 [00:33<00:00,  5.57s/it]


{'faithfulness': 0.8462, 'context_recall': 0.8333, 'answer_relevancy': 0.9703}

In [37]:
result.to_pandas()


,question,answer,contexts,ground_truth,faithfulness,context_recall,answer_relevancy
0,What is self attention?,"Self-attention, also known as intra-attention,...","[Self-attention, sometimes called intra-attent...",Self-attention lets each token in a sequence a...,0.692308,0.666667,0.970265
1,Who wrote Attention Is All You Need?,"The paper ""Attention Is All You Need"" was writ...","[Provided proper attribution is provided, Goog...",Attention Is All You Need was written by Ashis...,1.000000,1.000000,NaN
